In [1]:
!pip install -q sentence-transformers faiss-cpu huggingface_hub gradio scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

np.random.seed(42)

FEATURES = ["Watch_Time", "Completion_Rate", "Replays", "Pauses", "Videos_Watched", "Quiz_Score"]

def load_engagement_data(n_students: int = 100) -> pd.DataFrame:
    data = {
        "Student_ID": [f"S{i:03d}" for i in range(1, n_students + 1)],
        "Watch_Time": np.random.randint(10, 100, n_students),
        "Completion_Rate": np.random.randint(20, 100, n_students),
        "Replays": np.random.randint(0, 8, n_students),
        "Pauses": np.random.randint(0, 10, n_students),
        "Videos_Watched": np.random.randint(1, 15, n_students),
        "Quiz_Score": np.random.randint(30, 100, n_students),
    }
    return pd.DataFrame(data)

def segment_students(df: pd.DataFrame, k: int = 3):
    X_scaled = StandardScaler().fit_transform(df[FEATURES])
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    df = df.copy()
    df["Cluster"] = kmeans.fit_predict(X_scaled)
    ranking = df.groupby("Cluster")["Quiz_Score"].mean().sort_values().index.tolist()
    level_names = ["Low Engagement", "Medium Engagement", "High Engagement"]
    cluster_to_label = {c: level_names[i] for i, c in enumerate(ranking)}
    df["Engagement_Level"] = df["Cluster"].map(cluster_to_label)
    return df

df = load_engagement_data()
segmented_df = segment_students(df)
segmented_df[["Student_ID", "Engagement_Level"]].head()


,Student_ID,Engagement_Level
0,S001,Low Engagement
1,S002,Medium Engagement
2,S003,Medium Engagement
3,S004,Medium Engagement
4,S005,Low Engagement


In [3]:
def get_student_segment(student_id: str) -> str | None:
    """Look up a student's engagement level. Returns None if not found."""
    student_id = student_id.strip().upper()
    row = segmented_df[segmented_df["Student_ID"] == student_id]
    if row.empty:
        return None
    return row.iloc[0]["Engagement_Level"]

print(get_student_segment("S001"))


Low Engagement


In [4]:
from google.colab import files

def load_transcript() -> str:
    print("Upload a lecture transcript (.txt). Skip to use the built-in demo transcript.")
    try:
        uploaded = files.upload()
    except Exception:
        uploaded = {}
    if uploaded:
        filename = next(iter(uploaded))
        return uploaded[filename].decode("utf-8")
    return """
Machine Learning is a subset of Artificial Intelligence that enables
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

There are three major types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labelled data to train a model. The model learns
the relationship between input features and known output labels.
Classification and regression are common supervised learning tasks.

Unsupervised learning works with unlabeled data. The objective is to find
hidden patterns or groups in the data. Clustering, such as K-Means, is a
common unsupervised learning technique that groups similar data points
together based on distance between them.

Reinforcement learning trains an agent to make decisions by rewarding
good actions and penalising bad ones, so the agent learns an optimal
strategy through trial and error.
"""

lecture_text = load_transcript()
print("Transcript length (chars):", len(lecture_text))


Upload a lecture transcript (.txt). Skip to use the built-in demo transcript.


Transcript length (chars): 958


In [5]:
import re

def chunk_text(text: str, chunk_size: int = 3, overlap: int = 1) -> list[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s]
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(sentences), step):
        chunk = " ".join(sentences[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

chunks = chunk_text(lecture_text)
print("Number of chunks:", len(chunks))


Number of chunks: 5


In [6]:
from sentence_transformers import SentenceTransformer
import faiss

class CourseGuideIndex:
    def __init__(self, chunks: list[str], model_name: str = "all-MiniLM-L6-v2"):
        self.chunks = chunks
        self.model = SentenceTransformer(model_name)
        emb = np.array(self.model.encode(chunks)).astype("float32")
        self.index = faiss.IndexFlatL2(emb.shape[1])
        self.index.add(emb)

    def retrieve(self, question: str, top_k: int = 3) -> list[str]:
        q_emb = np.array(self.model.encode([question])).astype("float32")
        _, idx = self.index.search(q_emb, top_k)
        return [self.chunks[i] for i in idx[0]]

    def get_context(self, question: str, top_k: int = 3) -> str:
        return "\n\n".join(self.retrieve(question, top_k))

course_index = CourseGuideIndex(chunks)
print("Indexed vectors:", course_index.index.ntotal)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed vectors: 5


In [7]:
from google.colab import userdata
from huggingface_hub import InferenceClient

HF_TOKEN = userdata.get("pblm1")
llm_client = InferenceClient(provider="auto", api_key=HF_TOKEN)
LLM_MODEL = "openai/gpt-oss-120b"
print("Token loaded:", HF_TOKEN is not None)


Token loaded: True


In [8]:
SEGMENT_STYLE = {
    "Low Engagement": (
        "The student has LOW engagement so far. Use very simple language, short sentences, "
        "give a concrete everyday example, and end with one encouraging sentence motivating "
        "them to keep watching the lectures."
    ),
    "Medium Engagement": (
        "The student has MEDIUM engagement. Give a clear, moderately detailed answer, and end "
        "with one practical tip for applying the concept."
    ),
    "High Engagement": (
        "The student has HIGH engagement. Give a more detailed, technical answer, and end by "
        "pointing them to a more advanced related concept worth exploring next."
    ),
}

# A tiny topic map used for the rule-based "what to study next" recommendation.
# Extend this with your real course's module list.
COURSE_TOPICS = {
    "Supervised Learning": ["supervised", "classification", "regression", "labelled"],
    "Unsupervised Learning": ["unsupervised", "clustering", "k-means", "unlabeled"],
    "Reinforcement Learning": ["reinforcement", "reward", "agent", "trial and error"],
}

def recommend_topic(question: str, engagement_level: str) -> str:
    """Rule-based recommendation: match the question to a known topic, then suggest a
    next step calibrated to the student's engagement level.
    """
    q_lower = question.lower()
    matched_topic = None
    for topic, keywords in COURSE_TOPICS.items():
        if any(kw in q_lower for kw in keywords):
            matched_topic = topic
            break
    matched_topic = matched_topic or "the core lecture material"

    if engagement_level == "Low Engagement":
        return f"Next step: rewatch the short intro section on '{matched_topic}' before moving on."
    elif engagement_level == "Medium Engagement":
        return f"Next step: try a practice question on '{matched_topic}' to check your understanding."
    else:
        return f"Next step: explore an advanced resource on '{matched_topic}' beyond the lecture."


In [9]:
def generate_personalized_answer(student_id: str, question: str, top_k: int = 3) -> dict:
    """Full Day 3 pipeline: segment lookup -> retrieval -> personalized prompt -> LLM ->
    answer + recommendation. Returns a dict so the Gradio app can display each part.
    """
    segment = get_student_segment(student_id)
    if segment is None:
        return {
            "segment": None,
            "answer": f"No student found with ID '{student_id}'.",
            "recommendation": "",
        }

    context = course_index.get_context(question, top_k=top_k)
    style_instruction = SEGMENT_STYLE[segment]

    prompt = f"""You are an AI course guide for students.

Use ONLY the lecture content below to answer the question.

Lecture Content:
{context}

Student Question:
{question}

Personalization instruction: {style_instruction}

If the answer is not present in the lecture, say
"I could not find this information in the lecture."
"""

    response = llm_client.chat_completion(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=220,
    )
    answer = response.choices[0].message.content
    recommendation = recommend_topic(question, segment)

    return {"segment": segment, "answer": answer, "recommendation": recommendation}


# Quick test — try the same question for a couple of different students to see the
# tailored tone/depth change with their engagement level.
for sid in ["S001", "S002"]:
    result = generate_personalized_answer(sid, "What is K-Means clustering?")
    print(f"--- {sid} ({result['segment']}) ---")
    print(result["answer"])
    print(result["recommendation"])
    print()


--- S001 (Low Engagement) ---
K‑Means clustering is a way to put similar things together without any labels.  
It looks at the distance between data points and groups the ones that are close to each other.  

**Everyday example:** Imagine you have a bunch of marbles of different colors and sizes. K‑Means would automatically sort them into piles, each pile containing marbles that look alike (e.g., all the big red ones together, all the small blue ones together).

Keep watching the lectures – you’re doing great!
Next step: rewatch the short intro section on 'Unsupervised Learning' before moving on.

--- S002 (Medium Engagement) ---
K‑Means is a **clustering** method—an unsupervised‑learning technique that groups data points that are similar to one another. It works by measuring the **distance** between points and assigning each point to the nearest cluster centre (called a “centroid”). As the algorithm iterates, the centroids are updated to better represent the groups, and points are re‑

In [10]:
import gradio as gr

def run_pipeline(student_id: str, question: str):
    if not student_id.strip():
        return "Enter a Student ID.", "", ""
    if not question.strip():
        return "Enter a question.", "", ""

    result = generate_personalized_answer(student_id, question)
    segment_text = result["segment"] or "Student not found"
    return segment_text, result["answer"], result["recommendation"]


with gr.Blocks(title="Personalized Course Guide") as demo:
    gr.Markdown("# 🎓 Personalized Course Guide (Day 3)")
    gr.Markdown("Answers and study recommendations adapt to the student's engagement segment.")

    sid_input = gr.Textbox(label="Student ID", placeholder="e.g. S001")
    q_input = gr.Textbox(label="Question", placeholder="e.g. What is K-Means clustering?")
    run_btn = gr.Button("Ask")

    segment_output = gr.Textbox(label="Engagement Segment")
    answer_output = gr.Textbox(label="Personalized Answer", lines=6)
    rec_output = gr.Textbox(label="Recommended Next Step")

    run_btn.click(run_pipeline, inputs=[sid_input, q_input],
                   outputs=[segment_output, answer_output, rec_output])

demo.launch(debug=False)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c6d8761da4c7ff40f9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
